In [1]:
import re
import regex
assert "regex" in str(regex.compile("(test){e<=1}"))
import pandas as pd
import os
import numpy as np
pd.set_option('display.max_colwidth', 40)
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
project_dir = Path().resolve()
os.chdir(project_dir)

In [2]:
df = pd.read_excel("hand_coded_sample.xlsx", header = 1)
print(len(df))
df = df.drop(df[df['shelly_notes'] == "Appeal"].index)
print(len(df))
df['snippet_clean'] = (
    df["complaint_clean"].fillna("").str.strip()
    + "\n"
    + df["findings_clean"].fillna("").str.strip()
).str.strip()

191
190


## Bias Complaint

### Training on Whole Snippet

In [3]:
df_train = df.dropna(subset=["bias_complaint"])
print(len(df_train))

190


In [4]:
X = df_train['snippet_clean']

In [5]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

In [6]:
# Bias training
y = df_train['bias_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'Bias complaint score:{accuracy_score(y_test, y_pred)}')

Bias complaint score:0.9473684210526315


In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["snippet_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["bias_complaint"] != df_train.iloc[j]["bias_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.index[i],
                    df_train.index[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 3 8
Potential coding inconsistency: 5 8
Potential coding inconsistency: 7 8
Potential coding inconsistency: 8 11
Potential coding inconsistency: 8 14
Potential coding inconsistency: 8 17
Potential coding inconsistency: 8 22
Potential coding inconsistency: 8 29
Potential coding inconsistency: 8 30
Potential coding inconsistency: 8 34
Potential coding inconsistency: 8 39
Potential coding inconsistency: 8 41
Potential coding inconsistency: 8 56
Potential coding inconsistency: 8 60
Potential coding inconsistency: 8 71


### Training on complaint only

In [8]:
df_train = df.dropna(subset=["bias_complaint"])
print(len(df_train))

190


In [9]:
X = df_train['complaint_clean']

In [10]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

In [11]:
# Bias training
y = df_train['bias_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'Bias complaint score:{accuracy_score(y_test, y_pred)}')

Bias complaint score:0.9473684210526315


In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["complaint_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["bias_complaint"] != df_train.iloc[j]["bias_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.index[i],
                    df_train.index[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 3 8
Potential coding inconsistency: 5 8
Potential coding inconsistency: 7 8
Potential coding inconsistency: 8 11
Potential coding inconsistency: 8 14
Potential coding inconsistency: 8 17
Potential coding inconsistency: 8 22
Potential coding inconsistency: 8 29
Potential coding inconsistency: 8 30
Potential coding inconsistency: 8 34
Potential coding inconsistency: 8 39
Potential coding inconsistency: 8 41
Potential coding inconsistency: 8 56
Potential coding inconsistency: 8 60
Potential coding inconsistency: 8 71
Potential coding inconsistency: 166 188


## Property Complaint

### Training on Whole Snippet

In [13]:
df_train = df.dropna(subset=["property_complaint"])
print(len(df_train))
X = df_train['snippet_clean']

190


In [14]:

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

# Property training
y = df_train['property_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'property complaint score:{accuracy_score(y_test, y_pred)}')

property complaint score:0.9210526315789473


In [15]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["snippet_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["property_complaint"] != df_train.iloc[j]["property_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 040-18
Potential coding inconsistency: 225-17 092-18
Potential coding inconsistency: 225-17 026-18
Potential coding inconsistency: 225-17 185-17
Potential coding inconsistency: 225-17 001-18
Potential coding inconsistency: 225-17 027-18
Potential coding inconsistency: 225-17 126-18
Potential coding inconsistency: 225-17 084-18
Potential coding inconsistency: 225-17 016-19
Potential coding inconsistency: 225-17 052-18
Potential coding inconsistency: 225-17 137-19
Potential coding inconsistency: 225-17 127-19
Potential coding inconsistency: 225-17 037-19
Potential coding inconsistency: 225-17 304-20
Potential coding inconsistency: 225-17 280-20
Potential coding inconsistency: 040-18 275-20
Potential coding inconsistency: 092-18 275-20
Potential coding inconsistency: 026-18 275-20
Potential coding inconsistency: 185-17 275-20
Potential coding inconsistency: 001-18 275-20
Potential coding inconsistency: 027-18 275-20
Potential coding inconsistency: 12

### Training on Complaint Only

In [16]:
df_train = df.dropna(subset=["property_complaint"])
print(len(df_train))
X = df_train['complaint_clean']

190


In [17]:

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

# Property training
y = df_train['property_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'property complaint score:{accuracy_score(y_test, y_pred)}')

property complaint score:0.9210526315789473


In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["complaint_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["property_complaint"] != df_train.iloc[j]["property_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 040-18
Potential coding inconsistency: 225-17 092-18
Potential coding inconsistency: 225-17 026-18
Potential coding inconsistency: 225-17 185-17
Potential coding inconsistency: 225-17 001-18
Potential coding inconsistency: 225-17 027-18
Potential coding inconsistency: 225-17 126-18
Potential coding inconsistency: 225-17 084-18
Potential coding inconsistency: 225-17 016-19
Potential coding inconsistency: 225-17 052-18
Potential coding inconsistency: 225-17 137-19
Potential coding inconsistency: 225-17 127-19
Potential coding inconsistency: 225-17 037-19
Potential coding inconsistency: 225-17 304-20
Potential coding inconsistency: 225-17 280-20
Potential coding inconsistency: 040-18 275-20
Potential coding inconsistency: 092-18 275-20
Potential coding inconsistency: 026-18 275-20
Potential coding inconsistency: 185-17 275-20
Potential coding inconsistency: 001-18 275-20
Potential coding inconsistency: 027-18 275-20
Potential coding inconsistency: 12

## Demeanor Complaint

### Training on Whole Snippet

In [19]:
df_train = df.dropna(subset=["demeanor_complaint"])
print(len(df_train))
X = df_train['snippet_clean']

190


In [20]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

# Demeanor training
y = df_train['demeanor_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'demeanor complaint score:{accuracy_score(y_test, y_pred)}')

demeanor complaint score:0.631578947368421


In [21]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["snippet_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["demeanor_complaint"] != df_train.iloc[j]["demeanor_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 040-18
Potential coding inconsistency: 225-17 185-17
Potential coding inconsistency: 225-17 027-18
Potential coding inconsistency: 225-17 052-18
Potential coding inconsistency: 225-17 127-19
Potential coding inconsistency: 225-17 304-20
Potential coding inconsistency: 225-17 280-20
Potential coding inconsistency: 040-18 092-18
Potential coding inconsistency: 040-18 026-18
Potential coding inconsistency: 040-18 001-18
Potential coding inconsistency: 040-18 126-18
Potential coding inconsistency: 040-18 084-18
Potential coding inconsistency: 040-18 016-19
Potential coding inconsistency: 040-18 137-19
Potential coding inconsistency: 040-18 037-19
Potential coding inconsistency: 040-18 275-20
Potential coding inconsistency: 092-18 185-17
Potential coding inconsistency: 092-18 027-18
Potential coding inconsistency: 092-18 052-18
Potential coding inconsistency: 092-18 127-19
Potential coding inconsistency: 092-18 304-20
Potential coding inconsistency: 09

### Training on Complaint Only

In [22]:
df_train = df.dropna(subset=["demeanor_complaint"])
print(len(df_train))
X = df_train['complaint_clean']

190


In [23]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

# Demeanor training
y = df_train['demeanor_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'demeanor complaint score:{accuracy_score(y_test, y_pred)}')

demeanor complaint score:0.6578947368421053


In [24]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["complaint_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["demeanor_complaint"] != df_train.iloc[j]["demeanor_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 160-18 212-19
Potential coding inconsistency: 225-17 040-18
Potential coding inconsistency: 225-17 185-17
Potential coding inconsistency: 225-17 027-18
Potential coding inconsistency: 225-17 052-18
Potential coding inconsistency: 225-17 127-19
Potential coding inconsistency: 225-17 304-20
Potential coding inconsistency: 225-17 280-20
Potential coding inconsistency: 040-18 092-18
Potential coding inconsistency: 040-18 026-18
Potential coding inconsistency: 040-18 001-18
Potential coding inconsistency: 040-18 126-18
Potential coding inconsistency: 040-18 084-18
Potential coding inconsistency: 040-18 016-19
Potential coding inconsistency: 040-18 137-19
Potential coding inconsistency: 040-18 037-19
Potential coding inconsistency: 040-18 275-20
Potential coding inconsistency: 092-18 185-17
Potential coding inconsistency: 092-18 027-18
Potential coding inconsistency: 092-18 052-18
Potential coding inconsistency: 092-18 127-19
Potential coding inconsistency: 09

## Regulatory Complaint

### Training on Whole Snippet

In [25]:
df_train = df.dropna(subset=["regulatory_complaint"])
print(len(df_train))
X = df_train['snippet_clean']

190


In [26]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

# Regulatory training
y = df_train['regulatory_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'Regulatory complaint score:{accuracy_score(y_test, y_pred)}')

Regulatory complaint score:0.7894736842105263


In [27]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["snippet_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["regulatory_complaint"] != df_train.iloc[j]["regulatory_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 092-18
Potential coding inconsistency: 225-17 052-18
Potential coding inconsistency: 225-17 037-19
Potential coding inconsistency: 040-18 092-18
Potential coding inconsistency: 040-18 052-18
Potential coding inconsistency: 040-18 037-19
Potential coding inconsistency: 092-18 026-18
Potential coding inconsistency: 092-18 185-17
Potential coding inconsistency: 092-18 027-18
Potential coding inconsistency: 092-18 126-18
Potential coding inconsistency: 092-18 084-18
Potential coding inconsistency: 092-18 137-19
Potential coding inconsistency: 092-18 127-19
Potential coding inconsistency: 092-18 304-20
Potential coding inconsistency: 092-18 275-20
Potential coding inconsistency: 092-18 280-20
Potential coding inconsistency: 026-18 052-18
Potential coding inconsistency: 026-18 037-19
Potential coding inconsistency: 185-17 052-18
Potential coding inconsistency: 185-17 037-19
Potential coding inconsistency: 001-18 052-18
Potential coding inconsistency: 00

### Training on Complaint Only

In [28]:
df_train = df.dropna(subset=["regulatory_complaint"])
print(len(df_train))
X = df_train['complaint_clean']

190


In [29]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

# Regulatory training
y = df_train['regulatory_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'Regulatory complaint score:{accuracy_score(y_test, y_pred)}')

Regulatory complaint score:0.7894736842105263


In [30]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["complaint_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["regulatory_complaint"] != df_train.iloc[j]["regulatory_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 092-18
Potential coding inconsistency: 225-17 052-18
Potential coding inconsistency: 225-17 037-19
Potential coding inconsistency: 040-18 092-18
Potential coding inconsistency: 040-18 052-18
Potential coding inconsistency: 040-18 037-19
Potential coding inconsistency: 092-18 026-18
Potential coding inconsistency: 092-18 185-17
Potential coding inconsistency: 092-18 027-18
Potential coding inconsistency: 092-18 126-18
Potential coding inconsistency: 092-18 084-18
Potential coding inconsistency: 092-18 137-19
Potential coding inconsistency: 092-18 127-19
Potential coding inconsistency: 092-18 304-20
Potential coding inconsistency: 092-18 275-20
Potential coding inconsistency: 092-18 280-20
Potential coding inconsistency: 026-18 052-18
Potential coding inconsistency: 026-18 037-19
Potential coding inconsistency: 185-17 052-18
Potential coding inconsistency: 185-17 037-19
Potential coding inconsistency: 001-18 052-18
Potential coding inconsistency: 00

## Weapon Complaint

### Training on Whole Snippet

In [31]:
df_train = df.dropna(subset=["weapon_complaint"])
print(len(df_train))
X = df_train['snippet_clean']

190


In [32]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

# Weapon training
y = df_train['weapon_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'Weapon complaint score:{accuracy_score(y_test, y_pred)}')

Weapon complaint score:0.9736842105263158


In [33]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["snippet_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["weapon_complaint"] != df_train.iloc[j]["weapon_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 040-18
Potential coding inconsistency: 040-18 092-18
Potential coding inconsistency: 040-18 026-18
Potential coding inconsistency: 040-18 185-17
Potential coding inconsistency: 040-18 001-18
Potential coding inconsistency: 040-18 027-18
Potential coding inconsistency: 040-18 126-18
Potential coding inconsistency: 040-18 084-18
Potential coding inconsistency: 040-18 016-19
Potential coding inconsistency: 040-18 052-18
Potential coding inconsistency: 040-18 137-19
Potential coding inconsistency: 040-18 127-19
Potential coding inconsistency: 040-18 037-19
Potential coding inconsistency: 040-18 304-20
Potential coding inconsistency: 040-18 275-20
Potential coding inconsistency: 040-18 280-20


In [34]:
### Training on Complaint Only

In [35]:
df_train = df.dropna(subset=["weapon_complaint"])
print(len(df_train))
X = df_train['complaint_clean']

190


In [36]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

# Weapon training
y = df_train['weapon_complaint']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'Weapon complaint score:{accuracy_score(y_test, y_pred)}')

Weapon complaint score:0.9736842105263158


In [37]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["complaint_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["weapon_complaint"] != df_train.iloc[j]["weapon_complaint"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 040-18
Potential coding inconsistency: 040-18 092-18
Potential coding inconsistency: 040-18 026-18
Potential coding inconsistency: 040-18 185-17
Potential coding inconsistency: 040-18 001-18
Potential coding inconsistency: 040-18 027-18
Potential coding inconsistency: 040-18 126-18
Potential coding inconsistency: 040-18 084-18
Potential coding inconsistency: 040-18 016-19
Potential coding inconsistency: 040-18 052-18
Potential coding inconsistency: 040-18 137-19
Potential coding inconsistency: 040-18 127-19
Potential coding inconsistency: 040-18 037-19
Potential coding inconsistency: 040-18 304-20
Potential coding inconsistency: 040-18 275-20
Potential coding inconsistency: 040-18 280-20


## Traffic Scenario

### Training on Whole Snippet

In [38]:
df_train = df.dropna(subset=["traffic_scenario"])
print(len(df_train))
X = df_train['snippet_clean']

189


In [39]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

y = df_train['traffic_scenario']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'Traffic scenario score:{accuracy_score(y_test, y_pred)}')

Traffic scenario score:0.8157894736842105


In [40]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["snippet_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["traffic_scenario"] != df_train.iloc[j]["traffic_scenario"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 027-18
Potential coding inconsistency: 225-17 084-18
Potential coding inconsistency: 225-17 127-19
Potential coding inconsistency: 225-17 275-20
Potential coding inconsistency: 040-18 027-18
Potential coding inconsistency: 040-18 084-18
Potential coding inconsistency: 040-18 127-19
Potential coding inconsistency: 040-18 275-20
Potential coding inconsistency: 092-18 027-18
Potential coding inconsistency: 092-18 084-18
Potential coding inconsistency: 092-18 127-19
Potential coding inconsistency: 092-18 275-20
Potential coding inconsistency: 026-18 027-18
Potential coding inconsistency: 026-18 084-18
Potential coding inconsistency: 026-18 127-19
Potential coding inconsistency: 026-18 275-20
Potential coding inconsistency: 185-17 027-18
Potential coding inconsistency: 185-17 084-18
Potential coding inconsistency: 185-17 127-19
Potential coding inconsistency: 185-17 275-20
Potential coding inconsistency: 001-18 027-18
Potential coding inconsistency: 00

### Training on Complaint Only

In [41]:
df_train = df.dropna(subset=["traffic_scenario"])
print(len(df_train))
X = df_train['complaint_clean']

189


In [42]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2
)

y = df_train['traffic_scenario']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb = MultinomialNB()

nb.fit(
    X_train_vec,
    y_train
)

y_pred = nb.predict(X_test_vec)

print(f'Traffic scenario score:{accuracy_score(y_test, y_pred)}')

Traffic scenario score:0.8157894736842105


In [43]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    df_train["complaint_clean"].tolist(),
    show_progress_bar=True
)

similarity = cosine_similarity(embeddings)

np.fill_diagonal(similarity, 0)

threshold = 0.85

for i in range(len(df_train)):
    for j in range(i+1, len(df_train)):
        if similarity[i,j] > threshold:
            if df_train.iloc[i]["traffic_scenario"] != df_train.iloc[j]["traffic_scenario"]:
                print(
                    "Potential coding inconsistency:",
                    df_train.case_number.iloc[i],
                    df_train.case_number.iloc[j]
                )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Potential coding inconsistency: 225-17 027-18
Potential coding inconsistency: 225-17 084-18
Potential coding inconsistency: 225-17 127-19
Potential coding inconsistency: 225-17 275-20
Potential coding inconsistency: 040-18 027-18
Potential coding inconsistency: 040-18 084-18
Potential coding inconsistency: 040-18 127-19
Potential coding inconsistency: 040-18 275-20
Potential coding inconsistency: 092-18 027-18
Potential coding inconsistency: 092-18 084-18
Potential coding inconsistency: 092-18 127-19
Potential coding inconsistency: 092-18 275-20
Potential coding inconsistency: 026-18 027-18
Potential coding inconsistency: 026-18 084-18
Potential coding inconsistency: 026-18 127-19
Potential coding inconsistency: 026-18 275-20
Potential coding inconsistency: 185-17 027-18
Potential coding inconsistency: 185-17 084-18
Potential coding inconsistency: 185-17 127-19
Potential coding inconsistency: 185-17 275-20
Potential coding inconsistency: 001-18 027-18
Potential coding inconsistency: 00